# 基于pytorch的高层API实现简单softmax回归（分类）模型
* 任务描述：</br>
将“线性模型-上.ipynb” 中的生成样本数据的`make_multiclass_classification`函数改为用pytorch实现并命名为`make_multiclass_classification_torch`，利用该函数按要求生成带噪音的样本，并采用pytorch的高层API来实现softmax回归（分类）。

In [ ]:
import datetime
import os

def printLog(name, ID):
    # 打印和
    print(': %s; : %s' % (name, ID))
    # 打印操作系统的用户名
    print("系统用户名:", os.getlogin())
    # 打印当前时间
    print('运行时间:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

# 0.相关函数学习
## 0.1 交叉熵损失函数：torch.nn.CrossEntropyLoss()
`torch.nn.CrossEntropyLoss()` 主要用于计算分类问题中的交叉熵损失。它适用于单标签分类问题，即每个样本只属于一个类别。</br>
它结合了 `nn.LogSoftmax()` 和 `nn.NLLLoss()` （负对数似然损失）的功能，并自动应用了 softmax 函数。这意味着，你只需要提供模型的原始输出（未经过 softmax 处理），CrossEntropyLoss 会自动完成 softmax 和损失计算。</br>
下面是一个使用`torch.nn.CrossEntropyLoss()` 的简单示例。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 假设我们有一些模型输出和真实标签
# 实际应用时，input是模型的预测结果，此处代表2个样本的预测结果，
# 每个预测结果有3列数据分别为属于类别0、1、2的概率
input = torch.tensor([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]], dtype=torch.float32)
print('input:', input)

# 真实标签，此处第1个样本的真实标签为2，第2个样本的真实标签为0
labels = torch.tensor([2, 0], dtype=torch.int64)
print('labels:', labels)

# 创建 CrossEntropyLoss 实例
criterion = nn.CrossEntropyLoss()

# 计算损失
loss = criterion(input, labels)

# 打印损失结果
print("Loss:", loss.item())

## 0.2 softmax函数：torch.nn.functional.softmax()

`torch.nn.functional.softmax()` 是 PyTorch 中用于对张量应用 softmax 函数的一个函数。softmax 函数通常用于将一组数值转换为概率分布，使得这些数值可以被解释为概率。它在多分类任务中非常有用，特别是在神经网络的输出层中，用于将原始输出转换为概率分布。<br/>
__定义__  <br/> 
`torch.nn.functional.softmax(input, dim=None, _stacklevel=3, dtype=None)`  <br/><br/>
__参数说明__  <br/>
* input (Tensor): 输入张量。
* dim (int): 指定沿哪个维度应用 softmax 函数。通常在分类任务中，dim 为类别维度（通常是最后一个维度，即 dim=-1 或 dim=1）。
* _stacklevel (int): 内部使用，用户一般不需要关心。
* dtype (torch.dtype): 输出张量的数据类型。如果指定了 dtype，则输出张量将具有此数据类型。默认情况下，输出张量具有与输入相同的类型。<br/>

__使用示例__  <br/>
以下是一个使用 `paddle.nn.functional.softmax` 函数的示例代码：

In [ ]:
import torch
import torch.nn.functional as F

# 创建一个输入 Tensor，假设有两个样本，每个样本有三个类别的 preds
preds = torch.tensor([[1.0, 2.0, 3.0], [1.0, -1.0, 0.0]], dtype=torch.float32)

# 使用 softmax 函数将 logits 转换为概率分布
probabilities = F.softmax(preds, dim=1)

# 打印转换后的概率分布
print("Probabilities:", probabilities)

# 1.构建数据集
我们首先构建一个简单的多分类任务，并构建训练集、验证集和测试集。 本任务的数据来自3个不同的簇，每个簇对一个类别。我们采集1000条样本，每个样本包含2个特征。

数据集的构建函数`make_multiclass_classification_torch`的代码实现如下：

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def make_multiclass_classification_torch(n_samples=100, n_features=2, n_classes=3, shuffle=True, noise=0.1):
    n_samples_per_class = [n_samples // n_classes] * n_classes
    for i in range(n_samples % n_classes):
        n_samples_per_class[i] += 1
    X = []
    y = []

    for class_id in range(n_classes):
        center = [class_id * 2.0, class_id * 1.5]

        X_class = np.random.randn(n_samples_per_class[class_id], n_features)
        X_class = X_class + center  # 平移数据到中心点
 
        if noise is not None:
            X_class += np.random.normal(0, noise, X_class.shape)
        y_class = np.full((n_samples_per_class[class_id], 1), class_id)
        X.append(X_class)
        y.append(y_class)
   
    X = np.vstack(X)
    y = np.vstack(y)
 
    if shuffle:
        indices = np.random.permutation(len(X))
        X = X[indices]
        y = y[indices]

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.int64)
    return X, y  
    
# 测试函数
X, y = make_multiclass_classification_torch(n_samples=100, n_features=2, n_classes=3, shuffle=True, noise=0.1)
print(X.shape, y.shape)

随机采集1000个样本，并进行可视化。

In [ ]:
# 设置随机种子以保持每次运行结果一致
torch.manual_seed(102)

# 采样1000个样本
n_samples = 1000
X, y = make_multiclass_classification_torch(n_samples=n_samples, n_features=2, n_classes=3, noise=0.2)

# 可视化生产的数据集，不同颜色代表不同类别
plt.figure(figsize=(5,5))
plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), marker='*', c=y.numpy())  # , cmap='viridis'
plt.title('Multiclass Classification Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.savefig('linear-dataset-vis2.pdf')
plt.show()

将实践数据拆分成训练集、验证集和测试集。其中训练集640条、验证集160条、测试集200条。

In [ ]:
num_train = 640
num_dev = 160
num_test = 200

X_train, y_train = X[:num_train], y[:num_train]
X_dev, y_dev = X[num_train:num_train + num_dev], y[num_train:num_train + num_dev]
X_test, y_test = X[num_train + num_dev:], y[num_train + num_dev:]

# 打印X_train和y_train的维度
print("X_train shape: ", X_train.shape, "y_train shape: ", y_train.shape)

# 打印前5个数据的标签
print('前5个数据的标签：', y_train[:5])

# 2. 基于Pytorch高层API的softmax回归（分类）模型定义
模型组网是深度学习任务中的重要一环，该环节定义了神经网络的层次结构、数据从输入到输出的计算过程（即前向计算）等。</br>
PyTorch 提供了灵活的方式来定义模型，包括：</br>
* 直接使用内置模型：
PyTorch 提供了许多内置的预训练模型，可以直接使用或者作为基础来进行修改。例如，`torchvision.models` 包含了许多常用的计算机视觉模型。</br>
* 使用 `torch.nn.Sequential` 组网：
如果模型是一个简单的顺序堆叠的层，可以使用 `torch.nn.Sequential` 容器来定义模型。这种情况下，只需要将所有层按顺序添加到 Sequential 对象中即可。</br>
* 使用 `torch.nn.Module` 组网：
对于更复杂的模型，通常需要继承 `torch.nn.Module` 类并定义自己的 forward 方法来指定模型的前向计算逻辑。</br>

下面`SoftmaxRegressionModel`模型的定义采用的是 `torch.nn.Module` 组网方式，组网包括三个步骤：
* 创建一个继承自 `torch.nn.Module` 的类；
* 在类的构造函数 __init__ 中定义组网用到的神经网络层（layer）；
* 在类的前向计算函数 forward 中使用定义好的 layer 执行前向计算。

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
 
class SoftmaxRegressionModel(nn.Module):
    def __init__(self, num_features=None, num_classes=3):
        super(SoftmaxRegressionModel, self).__init__()
        
        if num_features is None:
            raise ValueError("num_features must be specified")
        self.linear = nn.Linear(num_features, num_classes)
    
    def forward(self, x):
        logits = self.linear(x)
        # 应用softmax函数得到类别概率
        outputs = F.softmax(logits, dim=1)    
        return outputs

# 3. 基于Pytorch高层API的模型训练

In [ ]:
train_labels = y_train.to(dtype=torch.int64)  # 直接将标签转换为int64
dev_labels = y_dev.to(dtype=torch.int64)  # 直接将标签转换为int64
test_labels = y_test.to(dtype=torch.int64)  # 直接将标签转换为int64  
model = SoftmaxRegressionModel(num_features=2, num_classes=3)

criterion = nn.CrossEntropyLoss()  # 交叉熵损失函数，内部会自动处理softmax
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Adam优化器

num_epochs = 800
for epoch in range(num_epochs):
    model.train()
    predictions = model(X_train)
    loss = criterion(predictions, train_labels.squeeze())  # squeeze()将形状从[n,1]变为[n]

    optimizer.zero_grad()  # 清空梯度
    loss.backward()        # 反向传播计算梯度
    optimizer.step()       # 更新参数

    with torch.no_grad():
        pred_classes = torch.argmax(predictions, dim=1)
        correct = (pred_classes == train_labels.squeeze()).float()
        acc = correct.sum() / len(correct)

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}, Acc: {acc.item():.4f}")

print("\nModel Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    test_pred_classes = torch.argmax(test_predictions, dim=1)
    
print("\n测试集预测类别:", test_pred_classes)
print("测试集真实类别:", test_labels.squeeze())

test_correct = (test_pred_classes == test_labels.squeeze()).float()
test_acc = test_correct.sum() / len(test_correct)
print(f"模型在测试集上的精度Acc: {test_acc.item():.4f}")

In [ ]:
# 打印训练后的最优模型参数
print("Model Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.detach().numpy()}")
    if name == 'linear.weight':  # 权值w
        w = list(param.detach().numpy())
    if name == 'linear.bias':    # 偏置b
        b = list(param.detach().numpy())

In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )

# 4.模型评估

In [ ]:
# 4.模型评估
model.eval()

with torch.no_grad():
    test_predictions = model(X_test)
    test_pred_classes = torch.argmax(test_predictions, dim=1)
    test_correct = (test_pred_classes == test_labels.squeeze()).float()
    test_acc = test_correct.sum() / len(test_correct)
    test_loss = criterion(test_predictions, test_labels.squeeze())

print("=" * 50)
print("模型评估结果")
print("=" * 50)
print(f"测试集损失: {test_loss.item():.6f}")
print(f"测试集准确率: {test_acc.item():.4f}")
print(f"测试集正确样本数: {test_correct.sum().int().item()}/{len(test_labels)}")

print("\n预测结果对比:")
print("预测类别:", test_pred_classes.numpy())
print("真实类别:", test_labels.squeeze().numpy())

# 尝试使用sklearn和seaborn进行详细评估
try:
    from sklearn.metrics import confusion_matrix, classification_report
    import seaborn as sns
    
    y_true = test_labels.squeeze().numpy()
    y_pred = test_pred_classes.numpy()

    cm = confusion_matrix(y_true, y_pred)
    print("\n混淆矩阵:")
    print(cm)

    print("\n详细分类报告:")
    print(classification_report(y_true, y_pred, target_names=[f'Class {i}' for i in range(3)]))

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[f'Class {i}' for i in range(3)],
                yticklabels=[f'Class {i}' for i in range(3)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

except ImportError:
    # 如果sklearn或seaborn未安装，使用手动实现
    print("sklearn或seaborn未安装，使用简化版评估...")
    
    # 手动计算混淆矩阵
    def manual_confusion_matrix(y_true, y_pred, n_classes=3):
        cm = torch.zeros(n_classes, n_classes, dtype=torch.int64)
        for t, p in zip(y_true, y_pred):
            cm[t, p] += 1
        return cm
    
    y_true = test_labels.squeeze()
    y_pred = test_pred_classes
    cm = manual_confusion_matrix(y_true, y_pred)
    
    print("\n混淆矩阵:")
    print("    预测:  0    1    2")
    print("真实:")
    for i in range(3):
        print(f"  {i}:    {cm[i, 0].item():2d}   {cm[i, 1].item():2d}   {cm[i, 2].item():2d}")
    
    # 使用matplotlib绘制混淆矩阵
    plt.figure(figsize=(6, 5))
    plt.imshow(cm.numpy(), cmap='Blues', interpolation='nearest')
    plt.colorbar()
    for i in range(3):
        for j in range(3):
            plt.text(j, i, str(cm[i, j].item()),
                    horizontalalignment="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    plt.xticks(range(3), ['Class 0', 'Class 1', 'Class 2'])
    plt.yticks(range(3), ['Class 0', 'Class 1', 'Class 2'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

# 计算每个类别的准确率
class_acc = []
y_true_np = test_labels.squeeze().numpy()
y_pred_np = test_pred_classes.numpy()

for i in range(3):
    class_mask = (y_true_np == i)
    if class_mask.sum() > 0:
        class_correct = (y_pred_np[class_mask] == y_true_np[class_mask]).sum()
        class_total = class_mask.sum()
        class_accuracy = class_correct / class_total
        class_acc.append(class_accuracy)
        print(f"Class {i} 准确率: {class_accuracy:.4f} ({class_correct}/{class_total})")

print(f"\n模型在测试集上的总体表现: {test_acc.item():.4f}")

def plot_decision_boundary(model, X, y):
    # 创建网格点
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    # 预测网格点的类别
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    grid_tensor = torch.tensor(grid_points, dtype=torch.float32)
    
    model.eval()
    with torch.no_grad():
        grid_pred = model(grid_tensor)
        grid_pred_class = torch.argmax(grid_pred, axis=1).numpy()
    
    # 绘制决策边界
    grid_pred_class = grid_pred_class.reshape(xx.shape)
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, grid_pred_class, alpha=0.3, cmap=plt.cm.Spectral)
    
    # 绘制数据点 - 修正这里：y已经是numpy数组，直接使用
    colors = ['red', 'blue', 'green']
    for i in range(3):
        idx = (y == i)  # y已经是numpy数组，直接比较
        plt.scatter(X[idx, 0], X[idx, 1], 
                    c=colors[i], label=f'Class {i}', 
                    edgecolors='black', s=50)
    
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Softmax Regression Decision Boundary')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 绘制决策边界 - 确保传入numpy数组
plot_decision_boundary(model, X_test.numpy(), test_labels.squeeze().numpy())

# 5.结果可视化

可视化观察类别划分结果。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 8))

# 将数据转换为numpy数组
X_test_np = X_test.numpy()
test_labels_np = test_labels.numpy().flatten()
test_pred_classes_np = test_pred_classes.numpy()

# 创建网格点用于绘制决策边界
x_min, x_max = X_test_np[:, 0].min() - 0.5, X_test_np[:, 0].max() + 0.5
y_min, y_max = X_test_np[:, 1].min() - 0.5, X_test_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

# 预测网格点的类别
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_tensor = torch.tensor(grid_points, dtype=torch.float32)

model.eval()
with torch.no_grad():
    grid_pred = model(grid_tensor)
    grid_pred_class = torch.argmax(grid_pred, axis=1).numpy()

# 绘制决策边界
grid_pred_class = grid_pred_class.reshape(xx.shape)
plt.contourf(xx, yy, grid_pred_class, alpha=0.3, cmap=plt.cm.Spectral)

# 绘制测试数据点
colors = ['red', 'blue', 'green']
for i in range(3):
    idx = test_labels_np == i
    plt.scatter(X_test_np[idx, 0], X_test_np[idx, 1], 
                c=colors[i], label=f'类别 {i}', 
                edgecolors='black', s=50)

plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('Softmax回归分类结果')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

cm = confusion_matrix(test_labels_np, test_pred_classes_np)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['类别0', '类别1', '类别2'],
            yticklabels=['类别0', '类别1', '类别2'])
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.tight_layout()
plt.show()

print("各类别准确率:")
class_acc = []
for i in range(3):
    class_mask = (test_labels_np == i)
    if class_mask.sum() > 0:
        class_correct = (test_pred_classes_np[class_mask] == test_labels_np[class_mask]).sum()
        class_total = class_mask.sum()
        class_accuracy = class_correct / class_total
        class_acc.append(class_accuracy)
        print(f"Class {i} 准确率: {class_accuracy:.4f} ({class_correct}/{class_total})")

print(f"\n模型在测试集上的总体表现: {test_acc.item():.4f}")
print("可视化完成！")

In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )